In [57]:
import transformers
from tokenizers import Tokenizer

## Load training data

In [6]:
train_files = ["/srv/data/lt2326-h26/a1/train/en.txt", 
                "/srv/data/lt2326-h26/a1/train/tr.txt", 
                "/srv/data/lt2326-h26/a1/train/zh.txt"]

## Train BPE tokenizers

In [7]:
from tokenizers.trainers import BpeTrainer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import UnicodeScripts

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = UnicodeScripts()

### Small BPE tokenizer

In [8]:
small_trainer = BpeTrainer(vocab_size=2500, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])

In [9]:
tokenizer.train(train_files, small_trainer)
tokenizer.save("trained_tokenizers/small_bpe_tokenizer.json")

### Large BPE tokenizer

In [10]:
big_trainer = BpeTrainer(vocab_size=10000, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])

In [11]:
tokenizer.train(train_files, big_trainer)
tokenizer.save("trained_tokenizers/large_bpe_tokenizer.json")

### Train a character-based tokenizer

In [12]:
from tokenizers.trainers import WordLevelTrainer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Split
from tokenizers import Regex

tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Split(Regex("."), behavior="isolated") #split on unicode character and make every character its own token ("isolated")

In [13]:
char_trainer = WordLevelTrainer(vocab_size=10000, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])

In [14]:
tokenizer.train(train_files, char_trainer)
tokenizer.save("trained_tokenizers/char_tokenizer.json")

## Analyze the tokenizers

In [15]:
small_bpe_tokenizer = Tokenizer.from_file("trained_tokenizers/small_bpe_tokenizer.json")

In [16]:
output = small_bpe_tokenizer.encode("coding at this ungodly hour?")
print(output.tokens)

['c', 'o', 'd', 'i', 'n', 'g', ' ', 'a', 't', ' ', 't', 'h', 'i', 's', ' ', 'u', 'n', 'g', 'o', 'd', 'l', 'y', ' ', 'h', 'o', 'u', 'r', '?']


In [17]:
large_bpe_tokenizer = Tokenizer.from_file("trained_tokenizers/large_bpe_tokenizer.json")

In [18]:
output = large_bpe_tokenizer.encode("coding at this ungodly hour?")
print(output.tokens)

['co', 'd', 'ing ', 'at ', 'th', 'is ', 'un', 'g', 'od', 'ly ', 'h', 'our', '?']


In [19]:
char_tokenizer = Tokenizer.from_file("trained_tokenizers/char_tokenizer.json")

In [20]:
print(char_tokenizer.token_to_id("[UNK]")) 

0


In [48]:
output = char_tokenizer.encode("coding at this ungodly hour?")
print(output.tokens)

['c', 'o', 'd', 'i', 'n', 'g', ' ', 'a', 't', ' ', 't', 'h', 'i', 's', ' ', 'u', 'n', 'g', 'o', 'd', 'l', 'y', ' ', 'h', 'o', 'u', 'r', '?']


### Validation set

In [46]:
val_files = ["/srv/data/lt2326-h26/a1/valid/en.txt", 
            "/srv/data/lt2326-h26/a1/valid/tr.txt", 
            "/srv/data/lt2326-h26/a1/valid/zh.txt"]

tokenizers = {"small_bpe_tokenizer" : small_bpe_tokenizer,
              "large_bpe_tokenizer" : large_bpe_tokenizer,
              "char_tokenizer": char_tokenizer}

In [62]:
val_lines = {}
val_encodings = {}
file_unicode_chars = {}

# Encoding(num_tokens=99, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

for key,value in tokenizers.items():
    print("Tokenizer: ", key)
    print("Vocab size: ", value.get_vocab_size())
    for file in val_files:
        with open(f"{file}", encoding="utf-8") as f:
            val_lines[file] = [line.strip() for line in f if line.strip()]
        file_unicode_chars[file] = sum(len(line) for line in val_lines[file])
        val_encodings[file] = value.encode_batch(val_lines[file], add_special_tokens=False)
    
    for file in val_encodings:
        print(file)
        num_sent = len(val_encodings[file])
        total_num_tokens = 0
        for i in range(len(val_encodings[file])):
            num_tokens = len(val_encodings[file][i])
            total_num_tokens += num_tokens
        print("Total num tokens: ", total_num_tokens)
        print("Avg num tokens/sent: ", total_num_tokens/num_sent)
        print("Avg num chars/token: ", file_unicode_chars[file]/total_num_tokens)
    print("-----------------------------------------------------------------------")

Tokenizer:  small_bpe_tokenizer
Vocab size:  9448
/srv/data/lt2326-h26/a1/valid/en.txt
Total num tokens:  368269
Avg num tokens/sent:  81.11651982378855
Avg num chars/token:  1.0
/srv/data/lt2326-h26/a1/valid/tr.txt
Total num tokens:  368383
Avg num tokens/sent:  98.62998661311914
Avg num chars/token:  1.0
/srv/data/lt2326-h26/a1/valid/zh.txt
Total num tokens:  368266
Avg num tokens/sent:  38.545740004186726
Avg num chars/token:  1.0
-----------------------------------------------------------------------
Tokenizer:  large_bpe_tokenizer
Vocab size:  10000
/srv/data/lt2326-h26/a1/valid/en.txt
Total num tokens:  174233
Avg num tokens/sent:  38.377312775330395
Avg num chars/token:  2.113658147423278
/srv/data/lt2326-h26/a1/valid/tr.txt
Total num tokens:  180268
Avg num tokens/sent:  48.26452476572958
Avg num chars/token:  2.0435296336565556
/srv/data/lt2326-h26/a1/valid/zh.txt
Total num tokens:  342030
Avg num tokens/sent:  35.79966506175424
Avg num chars/token:  1.0767067216326054
-------